In [6]:
import  pandas as pd
import numpy as np
import json
from pandas import json_normalize
import matplotlib.pyplot as plt
pd.set_option('future.no_silent_downcasting', True)
import matplotlib.ticker as mticker
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import re
from datetime import datetime
import requests


In [8]:
BASE_URL = "http://localhost:1234/api/v1/chat"

In [30]:
prompt_template = """Extract the features from this real estate description.
Return ONLY a valid JSON object, nothing else.

Features:
    "surface",
    "pieces",
    "etage",
    
    # -------------------------
    # 6. Property Quality
    # -------------------------
    "standing",
    "annee_constr",

    # -------------------------
    # 7. Nearby Amenities
    # -------------------------
    "ecole",
    "pharmacie",
    "hopital",
    "marche",
    "magasin",
    "restaurant",
    "bus",
    "railway",
    
    
    "caracteristiques", // A list of additional features mentioned in the description
    RQ: If a feature is not mentioned in the description, set its value to null.
    Its not necessary to fill all the features, only those that are mentioned in the description.
    do not try to guess the value of a feature if its not mentioned in the description, just set it to null.
    do not add any additional features that are not mentioned in the description, just extract precisely those that are mentioned.
    put in the caracteristiques field any additional features that are mentioned in the description. do not ignore any additional feature that is mentioned in the description, put it in the caracteristiques field.

Description:
{description}

JSON:"""

In [31]:
df = pd.read_csv("dataset_clean.csv")

df["description"]

C:\Users\Mohamed\AppData\Local\Temp\ipykernel_26192\280328780.py:1: DtypeWarning: Columns (0,10,11,12,16,18,19,20,21,22,23,24,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("dataset_clean.csv")


0        Réf: AH077 Nabeul / Hammamet Appartement / à v...
1        Réf: MHS186 Nabeul / Hammamet Villa / à vendre...
2        Réf: MH112 Nabeul / Hammamet nord Duplex / à v...
3        Réf: AH079 Nabeul / Hammamet centre Appartemen...
4        Réf: THS156 Nabeul / Hammamet Terrain / à vend...
                               ...                        
50664    📍MENZAH 9 B : APPARTEMENT S+2  À LOUER \n\nL’a...
50665    L’agence #MON_RÈSEAU met à la location un rez-...
50666    📍 El MENZAH 6 À LOUER – VILLA S+7 USAGE BUREAU...
50667    L’agence #MON_RÉSEAU met à la location des bur...
50668    📍ENNASR 2 : APPARTEMENT S+1 RICHEMENT MEUBLÉ A...
Name: description, Length: 50669, dtype: object

In [32]:
desc=[]
results = []
for i, row in df.iterrows():
    print(f"Processing {i+1}/{len(df)}...")
    
    prompt = prompt_template.format(description=row["description"])
    desc.append(row["description"])
    response = requests.post(BASE_URL, json={
    "model": "meta-llama-3.1-8b-instruct",
    "input": prompt,
    "context_length": 8000,
    "temperature": 0
})
    print ("Response received.")
    print (response.text)
    raw = response.json()["output"][0]["content"].strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    
    try:
        results.append(json.loads(raw))
    except json.JSONDecodeError:
        results.append(None)
        
    if i==3:
        break
    
for i, res in enumerate(results):
    print(f"Description {i+1}:")
    print(desc[i])
    print(f"Result {i+1}:")
    print(json.dumps(res, indent=2))
    print("\n\n")

Processing 1/50669...
Response received.
{
  "model_instance_id": "meta-llama-3.1-8b-instruct:2",
  "output": [
    {
      "type": "message",
      "content": "{\n    \"surface\": 55,\n    \"pieces\": null,\n    \"etage\": null,\n    \"standing\": null,\n    \"annee_constr\": null,\n    \"ecole\": null,\n    \"pharmacie\": null,\n    \"hopital\": null,\n    \"marche\": null,\n    \"magasin\": null,\n    \"restaurant\": null,\n    \"bus\": null,\n    \"railway\": null,\n    \"caracteristiques\": {\n        \"Coquet studio à vendre à 140 md à hammamet\": null\n    }\n}"
    }
  ],
  "stats": {
    "input_tokens": 497,
    "total_output_tokens": 122,
    "reasoning_output_tokens": 0,
    "tokens_per_second": 7.850626515388097,
    "time_to_first_token_seconds": 1.453
  },
  "response_id": "resp_8e9226b37793857bb75940f00a03183d9411de8bc405b5bd"
}
Processing 2/50669...
Response received.
{
  "model_instance_id": "meta-llama-3.1-8b-instruct:2",
  "output": [
    {
      "type": "message",
 